In [1]:
#reading pdf
!pip install pypdf


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from pypdf import PdfReader

In [3]:
documents_folder = Path(r"C:\Users\reyha\Desktop\turin-mobility-ai\documents")

In [4]:
documents = []

for pdf_file in documents_folder.glob("*.pdf"):

    reader = PdfReader(pdf_file)

    for page_number, page in enumerate(reader.pages):

        text = page.extract_text()

        documents.append({
            "source": pdf_file.name,
            "page": page_number + 1,
            "text": text
        })

In [5]:
!pip install pypdf tiktoken


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:

from pypdf import PdfReader
import tiktoken


# Folder containing your PDFs
PDF_FOLDER = Path(r"C:\Users\reyha\Desktop\turin-mobility-ai\documents")


# Tokenizer
encoding = tiktoken.get_encoding("cl100k_base")


def extract_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


# Find all PDFs
pdf_files = list(PDF_FOLDER.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files\n")


for pdf_path in pdf_files:

    # Extract text
    text = extract_pdf_text(pdf_path)

    # Count words
    words = len(text.split())

    # Count tokens
    tokens = encoding.encode(text)
    token_count = len(tokens)

    print(f"PDF: {pdf_path.name}")
    print(f"Words: {words:,}")
    print(f"Tokens: {token_count:,}")
    print("-" * 50)

Found 3 PDF files

PDF: parking_rules.pdf
Words: 271
Tokens: 611
--------------------------------------------------
PDF: traffic_restrictions.pdf
Words: 905
Tokens: 1,988
--------------------------------------------------
PDF: ztl_rules.pdf
Words: 198
Tokens: 436
--------------------------------------------------


In [7]:
import re
import unicodedata


for pdf_path in pdf_files:

    # Extract raw text
    text = extract_pdf_text(pdf_path)

    print("=" * 80)
    print("PDF:", pdf_path.name)
    print("=" * 80)

    # 1. Basic information
    print("Characters:", len(text))
    print("Words:", len(text.split()))

    # 2. Check tabs
    print("Tabs \\t:", text.count("\t"))

    # 3. Check carriage returns
    print("Carriage returns \\r:", text.count("\r"))

    # 4. Check null characters
    print("Null characters \\x00:", text.count("\x00"))

    # 5. Check many spaces together
    multiple_spaces = re.findall(r" {2,}", text)
    print("Multiple-space problems:", len(multiple_spaces))

    # 6. Check many empty lines
    many_newlines = re.findall(r"\n{3,}", text)
    print("Many empty-line problems:", len(many_newlines))

    # 7. Check broken words such as:
    # restric-
    # tions
    broken_words = re.findall(r"\w+-\n\w+", text)
    print("Possible broken words:", len(broken_words))

    # Show some examples
    if broken_words:
        print("Examples:", broken_words[:5])

    # 8. Check lines that contain only page numbers
    lines = text.splitlines()

    page_number_lines = []

    for line in lines:
        if line.strip().isdigit():
            page_number_lines.append(line.strip())

    print("Possible page-number lines:", len(page_number_lines))

    if page_number_lines:
        print("Examples:", page_number_lines[:10])

    # 9. Check uppercase lines
    uppercase_lines = []

    for line in lines:

        line = line.strip()

        if len(line) > 3 and line.isupper():
            uppercase_lines.append(line)

    print("Uppercase lines:", len(uppercase_lines))

    if uppercase_lines:
        print("Examples:")
        for line in uppercase_lines[:10]:
            print("  ", line)

    # 10. Show first part of raw PDF text
    print("\nFIRST 1500 CHARACTERS:")
    print("-" * 50)
    print(text[:1500])

    print("\n")

PDF: parking_rules.pdf
Characters: 1713
Words: 271
Tabs \t: 0
Carriage returns \r: 0
Null characters \x00: 0
Multiple-space problems: 0
Many empty-line problems: 0
Possible broken words: 0
Possible page-number lines: 0
Uppercase lines: 3
Examples:
   SOSTA STRISCE BLU
   CON "GTT - SOSTAPP" PAGHI LA SOSTA AL MINUTO E ACQUISTI GLI
   ABBONAMENTI E I CARNET PER LA SOSTA NELLE STRISCE BLU

FIRST 1500 CHARACTERS:
--------------------------------------------------
 
SOSTA STRISCE BLU
Zona blu: mappa della sosta a
pagamento (pdf)
Tariffe e modalità di pagamento
della sosta 
Sosta per residenti, dimoranti e
altre categorie
Sosta per persone con disabilità 
Centri di Servizi al Cliente
Contravvenzioni e ricorsi
L’area delle strisce blu è suddivisa in quattro macro zone: la sosta nella ZTL Centrale costa € 2,80 l’ora, la sosta a
tariffa ordinaria € 1,70 l’ora, la sosta a tariffa ridotta € 1,50 l'ora e la sosta a tariffa smart € 1,20 l'ora.
La sosta a pagamento è in vigore dal lunedì al sabato, 

In [8]:
import re
import unicodedata


def clean_text(text):

    # 1. Normalize Unicode
    text = unicodedata.normalize("NFKC", text)

    # 2. Replace non-breaking spaces
    text = text.replace("\xa0", " ")

    # 3. Split text into lines
    lines = text.splitlines()

    clean_lines = []

    # Lines coming from website interface
   
    unwanted_lines = [
    "Argomenti",
    "Tempo di lettura",
    "INDICE DELLA PAGINA",
    "Descrizione",
    "Link utili",
    "Contatta il comune",
    "Problemi in città",
    "Unità organizzativa responsabile",
    "Leggi le domande frequenti",
    "Richiedi assistenza",
    "Prenota appuntamento",
    "Segnala disservizio",
    "Valuta da 1 a 5 stelle la pagina",
    "Quanto sono chiare le informazioni su questa pagina?"
]

    for line in lines:

        # Remove spaces at beginning/end
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Remove unwanted website lines
        if line in unwanted_lines:
            continue

        # Remove website navigation
        if line.startswith("Home /"):
            continue

        # Remove reading-time lines such as:
        # 4 minuti
        # 17 minuti
        if re.fullmatch(r"\d+\s+minuti", line):
            continue

        # Replace multiple spaces inside the line
        line = re.sub(r"\s+", " ", line)

        clean_lines.append(line)

    # Keep line structure
    text = "\n".join(clean_lines)

    return text.strip()

In [9]:
for pdf_path in pdf_files:

    raw_text = extract_pdf_text(pdf_path)

    cleaned_text = clean_text(raw_text)

    print("=" * 80)
    print("PDF:", pdf_path.name)

    print("\nBEFORE CLEANING:\n")
    print(raw_text[:1000])

    print("\nAFTER CLEANING:\n")
    print(cleaned_text[:1000])

    print("\n" + "=" * 80)

PDF: parking_rules.pdf

BEFORE CLEANING:

 
SOSTA STRISCE BLU
Zona blu: mappa della sosta a
pagamento (pdf)
Tariffe e modalità di pagamento
della sosta 
Sosta per residenti, dimoranti e
altre categorie
Sosta per persone con disabilità 
Centri di Servizi al Cliente
Contravvenzioni e ricorsi
L’area delle strisce blu è suddivisa in quattro macro zone: la sosta nella ZTL Centrale costa € 2,80 l’ora, la sosta a
tariffa ordinaria € 1,70 l’ora, la sosta a tariffa ridotta € 1,50 l'ora e la sosta a tariffa smart € 1,20 l'ora.
La sosta a pagamento è in vigore dal lunedì al sabato, dalle ore 8.00 alle ore 19.30.
Nelle seguenti aree ospedali il termine della sosta a pagamento è anticipato alle 18.30:
Oftalmico (vedi dettagli in scheda sottozona A1)
Valdese (vedi dettagli in scheda sottozona B1)
Maria Vittoria (vedi dettagli in scheda sottozona D2)
Molinette (vedi dettagli in scheda Bramante/Dogliotti)
Mauriziano (vedi dettagli in scheda sottozona C2)
San Giovanni Vecchio (vedi dettagli in schede s

In [10]:
import pandas as pd


# -----------------------------------
# Chunk settings
# -----------------------------------

CHUNK_SIZE = 100
CHUNK_OVERLAP =20


all_chunks = []


# -----------------------------------
# Create chunks from CLEANED text
# -----------------------------------

for pdf_path in pdf_files:

    # 1. Extract raw text from PDF
    raw_text = extract_pdf_text(pdf_path)

    # 2. Clean the text
    cleaned_text = clean_text(raw_text)

    # 3. Convert CLEANED text to tokens
    tokens = encoding.encode(cleaned_text)

    print("PDF:", pdf_path.name)
    print("Cleaned tokens:", len(tokens))

    start = 0
    chunk_number = 1


    # 4. Create chunks
    while start < len(tokens):

        end = start + CHUNK_SIZE

        # Take tokens for this chunk
        chunk_tokens = tokens[start:end]

        # Convert tokens back to readable text
        chunk_text = encoding.decode(chunk_tokens)

        # Save chunk
        all_chunks.append({
            "source": pdf_path.name,
            "chunk_number": chunk_number,
            "token_count": len(chunk_tokens),
            "text": chunk_text
        })

        # Move 350 tokens forward
        # 400 chunk - 50 overlap = 350
        start = start + CHUNK_SIZE - CHUNK_OVERLAP

        chunk_number = chunk_number + 1

    print("-" * 50)


# -----------------------------------
# Show total chunks
# -----------------------------------

print()
print("Total chunks:", len(all_chunks))

PDF: parking_rules.pdf
Cleaned tokens: 609
--------------------------------------------------
PDF: traffic_restrictions.pdf
Cleaned tokens: 1837
--------------------------------------------------
PDF: ztl_rules.pdf
Cleaned tokens: 335
--------------------------------------------------

Total chunks: 36


In [11]:
# -----------------------------------
# Convert chunks to a DataFrame
# -----------------------------------

chunks_df = pd.DataFrame(all_chunks)

print()
print(chunks_df[["source", "chunk_number", "token_count"]])


                      source  chunk_number  token_count
0          parking_rules.pdf             1          100
1          parking_rules.pdf             2          100
2          parking_rules.pdf             3          100
3          parking_rules.pdf             4          100
4          parking_rules.pdf             5          100
5          parking_rules.pdf             6          100
6          parking_rules.pdf             7          100
7          parking_rules.pdf             8           49
8   traffic_restrictions.pdf             1          100
9   traffic_restrictions.pdf             2          100
10  traffic_restrictions.pdf             3          100
11  traffic_restrictions.pdf             4          100
12  traffic_restrictions.pdf             5          100
13  traffic_restrictions.pdf             6          100
14  traffic_restrictions.pdf             7          100
15  traffic_restrictions.pdf             8          100
16  traffic_restrictions.pdf             9     

In [12]:
# -----------------------------------
# Save chunks to CSV
# -----------------------------------

OUTPUT_FILE = Path(
    r"C:\Users\reyha\Desktop\turin-mobility-ai\chunks.csv"
)

chunks_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print()
print("Chunks saved to:")
print(OUTPUT_FILE)


Chunks saved to:
C:\Users\reyha\Desktop\turin-mobility-ai\chunks.csv


In [13]:
for i in range(len(chunks_df)):

    print("Source:", chunks_df.loc[i, "source"])
    print("Chunk number:", chunks_df.loc[i, "chunk_number"])
    print("Tokens:", chunks_df.loc[i, "token_count"])

    print("\nTEXT:\n")
    print(chunks_df.loc[i, "text"])

    print("\n" + "-" * 80 + "\n")

Source: parking_rules.pdf
Chunk number: 1
Tokens: 100

TEXT:

SOSTA STRISCE BLU
Zona blu: mappa della sosta a
pagamento (pdf)
Tariffe e modalità di pagamento
della sosta
Sosta per residenti, dimoranti e
altre categorie
Sosta per persone con disabilità
Centri di Servizi al Cliente
Contravvenzioni e ricorsi
L’area delle strisce blu è suddivisa in quattro macro zone: la sosta

--------------------------------------------------------------------------------

Source: parking_rules.pdf
Chunk number: 2
Tokens: 100

TEXT:

’area delle strisce blu è suddivisa in quattro macro zone: la sosta nella ZTL Centrale costa € 2,80 l’ora, la sosta a
tariffa ordinaria € 1,70 l’ora, la sosta a tariffa ridotta € 1,50 l'ora e la sosta a tariffa smart € 1,20 l'ora.
La sosta a pagamento è in vigore dal lunedì

--------------------------------------------------------------------------------

Source: parking_rules.pdf
Chunk number: 3
Tokens: 100

TEXT:

1,20 l'ora.
La sosta a pagamento è in vigore dal lunedì al 

In [14]:
!pip install sentence-transformers


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

c:\Users\reyha\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13211.77it/s]


In [16]:
all_embeddings = []


for i in range(len(chunks_df)):

    # Get chunk text
    chunk_text = chunks_df.loc[i, "text"]

    # Create embedding
    embedding = model.encode(chunk_text)

    # Save embedding
    all_embeddings.append(embedding)

    print(
        "Created embedding for:",
        chunks_df.loc[i, "source"],
        "- Chunk",
        chunks_df.loc[i, "chunk_number"]
    )

Created embedding for: parking_rules.pdf - Chunk 1
Created embedding for: parking_rules.pdf - Chunk 2
Created embedding for: parking_rules.pdf - Chunk 3
Created embedding for: parking_rules.pdf - Chunk 4
Created embedding for: parking_rules.pdf - Chunk 5
Created embedding for: parking_rules.pdf - Chunk 6
Created embedding for: parking_rules.pdf - Chunk 7
Created embedding for: parking_rules.pdf - Chunk 8
Created embedding for: traffic_restrictions.pdf - Chunk 1
Created embedding for: traffic_restrictions.pdf - Chunk 2
Created embedding for: traffic_restrictions.pdf - Chunk 3
Created embedding for: traffic_restrictions.pdf - Chunk 4
Created embedding for: traffic_restrictions.pdf - Chunk 5
Created embedding for: traffic_restrictions.pdf - Chunk 6
Created embedding for: traffic_restrictions.pdf - Chunk 7
Created embedding for: traffic_restrictions.pdf - Chunk 8
Created embedding for: traffic_restrictions.pdf - Chunk 9
Created embedding for: traffic_restrictions.pdf - Chunk 10
Created emb

In [17]:
print("Number of chunks:", len(chunks_df))
print("Number of embeddings:", len(all_embeddings))

Number of chunks: 36
Number of embeddings: 36


In [18]:
import numpy as np

np.save(
    r"C:\Users\reyha\Desktop\turin-mobility-ai\embeddings.npy",
    np.array(all_embeddings)
)

print("Embeddings saved")

Embeddings saved


In [19]:
question = "How much does parking cost in the ZTL?"

In [20]:
question_embedding = model.encode(question)

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


# -----------------------------------
# 1. User question
# -----------------------------------

question = "How much does parking cost in the ZTL?"


# -----------------------------------
# 2. Create embedding for question
# -----------------------------------

question_embedding = model.encode(question)


# -----------------------------------
# 3. Compare question with all chunks
# -----------------------------------

similarities = cosine_similarity(
    [question_embedding],
    np.array(all_embeddings)
)[0]


# -----------------------------------
# 4. Add similarity to our table
# -----------------------------------

results_df = chunks_df.copy()

results_df["similarity"] = similarities


# -----------------------------------
# 5. Get top 3 most similar chunks
# -----------------------------------

top_results = results_df.sort_values(
    "similarity",
    ascending=False
).head(3)


# -----------------------------------
# 6. Show results
# -----------------------------------

for i, row in top_results.iterrows():

    print("Source:", row["source"])
    print("Chunk:", row["chunk_number"])
    print("Similarity:", round(row["similarity"], 3))

    print("\nText:")
    print(row["text"])

    print("\n" + "-" * 80 + "\n")

Source: parking_rules.pdf
Chunk: 2
Similarity: 0.565

Text:
’area delle strisce blu è suddivisa in quattro macro zone: la sosta nella ZTL Centrale costa € 2,80 l’ora, la sosta a
tariffa ordinaria € 1,70 l’ora, la sosta a tariffa ridotta € 1,50 l'ora e la sosta a tariffa smart € 1,20 l'ora.
La sosta a pagamento è in vigore dal lunedì

--------------------------------------------------------------------------------

Source: ztl_rules.pdf
Chunk: 2
Similarity: 0.529

Text:
TL Centrale
Recarsi occasionalmente nella ZTL Centrale
Recarsi in una scuola nella ZTL Centrale
Recarsi in una struttura sanitaria nella ZTL Centrale
ZTL Viabilità e parcheggi
Entrare in un parcheggio pubblico in struttura nella ZTL
Centrale
Scarica la mappa della ZTL in formato PDF
La normativa:
Deliberazione della Giunta Comunale mecc.

--------------------------------------------------------------------------------

Source: ztl_rules.pdf
Chunk: 1
Similarity: 0.525

Text:
ZTL Centrale
All'interno di questa pagina trove

In [22]:
question = "Quanto costa la sosta nella ZTL Centrale?"

In [23]:
question_embedding = model.encode(question)

similarities = cosine_similarity(
    [question_embedding],
    np.array(all_embeddings)
)[0]

results_df = chunks_df.copy()
results_df["similarity"] = similarities

top_results = results_df.sort_values(
    "similarity",
    ascending=False
).head(3)

for i, row in top_results.iterrows():

    print("Source:", row["source"])
    print("Chunk:", row["chunk_number"])
    print("Similarity:", round(row["similarity"], 3))

    print("\nText:")
    print(row["text"])

    print("\n" + "-" * 80 + "\n")

Source: parking_rules.pdf
Chunk: 2
Similarity: 0.708

Text:
’area delle strisce blu è suddivisa in quattro macro zone: la sosta nella ZTL Centrale costa € 2,80 l’ora, la sosta a
tariffa ordinaria € 1,70 l’ora, la sosta a tariffa ridotta € 1,50 l'ora e la sosta a tariffa smart € 1,20 l'ora.
La sosta a pagamento è in vigore dal lunedì

--------------------------------------------------------------------------------

Source: ztl_rules.pdf
Chunk: 1
Similarity: 0.597

Text:
ZTL Centrale
All'interno di questa pagina troverai tutte le informazioni relative
alla ZTL Centrale
Circolare nella Zona a Traffico Limitato (ZTL Centrale)
Esenzioni a posteriori
Esenzioni a priori
Altre limitazioni all'interno della ZTL Centrale
Altre limitazioni all'esterno della ZTL Centrale
Recarsi occasionalmente nella ZTL Centrale
Recarsi in una scu

--------------------------------------------------------------------------------

Source: ztl_rules.pdf
Chunk: 2
Similarity: 0.583

Text:
TL Centrale
Recarsi occasiona

In [24]:
def search_documents(question):

    # Convert question to embedding
    question_embedding = model.encode(question)

    # Compare with all document chunks
    similarities = cosine_similarity(
        [question_embedding],
        np.array(all_embeddings)
    )[0]

    # Copy chunk table
    results_df = chunks_df.copy()

    # Add similarity scores
    results_df["similarity"] = similarities

    # Select top 3 results
    top_results = results_df.sort_values(
        "similarity",
        ascending=False
    ).head(3)

    return top_results

In [25]:
results = search_documents(
    "How much does parking cost in the ZTL?"
)

print(results[["source", "chunk_number", "similarity"]])

               source  chunk_number  similarity
1   parking_rules.pdf             2    0.564791
32      ztl_rules.pdf             2    0.528755
31      ztl_rules.pdf             1    0.524743


In [26]:
results = search_documents(
    "Quanto costa la sosta nella ZTL Centrale?"
)
print(results[["source", "chunk_number", "similarity"]])

               source  chunk_number  similarity
1   parking_rules.pdf             2    0.708294
31      ztl_rules.pdf             1    0.597283
32      ztl_rules.pdf             2    0.583271


In [27]:
results = search_documents(
    "Which diesel vehicles are restricted?"
)
print(results[["source", "chunk_number", "similarity"]])

                      source  chunk_number  similarity
22  traffic_restrictions.pdf            15    0.635761
24  traffic_restrictions.pdf            17    0.523767
23  traffic_restrictions.pdf            16    0.517344


In [28]:
search_documents(question)

,source,chunk_number,token_count,text,similarity
1,parking_rules.pdf,2,100,’area delle strisce blu è suddivisa in quattro...,0.708294
31,ztl_rules.pdf,1,100,ZTL Centrale\nAll'interno di questa pagina tro...,0.597283
32,ztl_rules.pdf,2,100,TL Centrale\nRecarsi occasionalmente nella ZTL...,0.583271


In [29]:
def answer_question(question):

    # 1. Find the 3 most relevant chunks
    top_results = search_documents(question)

    # 2. Combine their text together
    context = "\n\n".join(
        top_results["text"].tolist()
    )

    # 3. Show the question
    print("Question:")
    print(question)

    # 4. Show the retrieved information
    print("\nRetrieved context:")
    print(context)

    return context

In [30]:
question = "How much does parking cost in the ZTL?"

answer_question(question)

Question:
How much does parking cost in the ZTL?

Retrieved context:
’area delle strisce blu è suddivisa in quattro macro zone: la sosta nella ZTL Centrale costa € 2,80 l’ora, la sosta a
tariffa ordinaria € 1,70 l’ora, la sosta a tariffa ridotta € 1,50 l'ora e la sosta a tariffa smart € 1,20 l'ora.
La sosta a pagamento è in vigore dal lunedì

TL Centrale
Recarsi occasionalmente nella ZTL Centrale
Recarsi in una scuola nella ZTL Centrale
Recarsi in una struttura sanitaria nella ZTL Centrale
ZTL Viabilità e parcheggi
Entrare in un parcheggio pubblico in struttura nella ZTL
Centrale
Scarica la mappa della ZTL in formato PDF
La normativa:
Deliberazione della Giunta Comunale mecc.

ZTL Centrale
All'interno di questa pagina troverai tutte le informazioni relative
alla ZTL Centrale
Circolare nella Zona a Traffico Limitato (ZTL Centrale)
Esenzioni a posteriori
Esenzioni a priori
Altre limitazioni all'interno della ZTL Centrale
Altre limitazioni all'esterno della ZTL Centrale
Recarsi occasional

"’area delle strisce blu è suddivisa in quattro macro zone: la sosta nella ZTL Centrale costa € 2,80 l’ora, la sosta a\ntariffa ordinaria € 1,70 l’ora, la sosta a tariffa ridotta € 1,50 l'ora e la sosta a tariffa smart € 1,20 l'ora.\nLa sosta a pagamento è in vigore dal lunedì\n\nTL Centrale\nRecarsi occasionalmente nella ZTL Centrale\nRecarsi in una scuola nella ZTL Centrale\nRecarsi in una struttura sanitaria nella ZTL Centrale\nZTL Viabilità e parcheggi\nEntrare in un parcheggio pubblico in struttura nella ZTL\nCentrale\nScarica la mappa della ZTL in formato PDF\nLa normativa:\nDeliberazione della Giunta Comunale mecc.\n\nZTL Centrale\nAll'interno di questa pagina troverai tutte le informazioni relative\nalla ZTL Centrale\nCircolare nella Zona a Traffico Limitato (ZTL Centrale)\nEsenzioni a posteriori\nEsenzioni a priori\nAltre limitazioni all'interno della ZTL Centrale\nAltre limitazioni all'esterno della ZTL Centrale\nRecarsi occasionalmente nella ZTL Centrale\nRecarsi in una scu"

In [52]:
def build_prompt(question, context):

    prompt = f"""
You are a Turin mobility assistant.

Use only the information provided in the context below.
Do not use outside knowledge or invent information.

Ignore context that is not relevant to the user's question.

If the context does not contain enough information to answer,
say that you do not have enough information.

Answer only what the user asks.
Answer in the same language as the user's question.
Keep the answer short, clear, and direct.

Context:
{context}

Question:
{question}

Give a short and direct answer.
"""

    return prompt

In [32]:
!pip install openai python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print(api_key is not None)

True


In [34]:
from openai import OpenAI
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)
print(client)

In [35]:
print(api_key.startswith("sk-or-"))
print(len(api_key))

True
73


In [36]:
question = "Explain RAG in one simple sentence."

response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": question
        }
    ]
)

answer = response.choices[0].message.content

print(answer)

RAG combines retrieving relevant documents with a language model to generate more accurate, context‑aware responses.


In [37]:
def answer_question(question):

    context = get_context(question)

    prompt = build_prompt(question, context)

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return answer

In [38]:
def answer_question(question):

    # Find relevant chunks
    top_results = search_documents(question)

    # Combine chunks
    context = "\n\n".join(
        top_results["text"].tolist()
    )

    # Build prompt
    prompt = build_prompt(question, context)

    # Send prompt to LLM
    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    # Get LLM answer
    answer = response.choices[0].message.content

    return answer

In [39]:
question = "How much does parking cost in the ZTL?"

answer = answer_question(question)

print(answer)

€2.80 per hour.


In [40]:
question = "Quanto costa la sosta nella ZTL Centrale?"
answer = answer_question(question)

print(answer)

€ 2,80 l’ora.


# AI Agent

In [41]:
import os
from dotenv import load_dotenv
import psycopg2

load_dotenv()

conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT")
)

print("Connected to PostgreSQL")

Connected to PostgreSQL


In [42]:
def data_tool(question):

    # Tell the LLM about our database
    schema = """
parking_locations:
name, id, status, total, free, tendance, lat, lng, collected_at

parking_observations:
observation_id, parking_id, status, free, tendance, collected_at

traffic_locations:
sensor_id, road_name, direction, lat, lng, period,
flow, speed, road_id, offset, collected_at

traffic_observations:
observation_id, sensor_id, direction, offset,
period, flow, speed, collected_at
"""

    # Ask the LLM to create SQL
    prompt = f"""
Write a PostgreSQL query to answer the user's question.

Database:

{schema}

Rules:
- Use only these four tables.
- Return only SQL.
- Use only SELECT or WITH.
- Do not modify the database.
- Ignore NULL values when comparing or ranking numeric values.
- For parking availability questions, use free IS NOT NULL.
- For current parking data, use parking_locations.
- For current traffic data, use traffic_locations.
- For historical data, use the observation tables.

Question:
{question}
"""

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    sql = response.choices[0].message.content.strip()

    # Remove markdown if the LLM adds it
    sql = sql.replace("```sql", "")
    sql = sql.replace("```", "")
    sql = sql.strip()

    print("Generated SQL:")
    print(sql)

    # Safety check
    if not (
        sql.upper().startswith("SELECT")
        or sql.upper().startswith("WITH")
    ):
        return "Unsafe SQL query."

    # Run SQL
    cursor = conn.cursor()
    cursor.execute(sql)

    rows = cursor.fetchall()

    columns = [
        column[0]
        for column in cursor.description
    ]

    cursor.close()

    # Combine column names with results
    data = [
        dict(zip(columns, row))
        for row in rows
    ]

    return data

In [43]:
def choose_tool(question):

    prompt = f"""
Choose the correct source for the user's question.

DOCUMENT:
Use for information from official documents:
- parking prices and tariffs
- parking hours
- ZTL rules
- traffic restrictions
- permits and regulations

DATA:
Use for information from PostgreSQL:
- available parking spaces
- parking occupancy
- traffic speed
- traffic flow
- historical traffic or parking data

BOTH:
Use when the question needs both
DOCUMENT and DATA.

UNKNOWN:
Use when the question cannot be answered
from the documents or PostgreSQL data.

Examples:

"How much does parking cost in the ZTL?"
DOCUMENT

"What is the average traffic speed?"
DATA

"How much does parking cost in the ZTL and what is the average traffic speed?"
BOTH

"What will the weather be tomorrow?"
UNKNOWN

Question:
{question}

Answer only:
DOCUMENT
DATA
BOTH
or
UNKNOWN
"""

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    decision = response.choices[0].message.content.strip()

    return decision

In [44]:
def split_question(question):

    prompt = f"""
Split the user's question into two parts.

DOCUMENT:
The part about rules, prices, ZTL, restrictions,
parking rules, permits or regulations.

DATA:
The part about traffic or parking measurements
stored in PostgreSQL.

Keep the same language as the user's question.

Question:
{question}

Return exactly:

DOCUMENT: ...
DATA: ...
"""

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    text = response.choices[0].message.content

    document_question = (
        text.split("DOCUMENT:")[1]
        .split("DATA:")[0]
        .strip()
    )

    data_question = (
        text.split("DATA:")[1]
        .strip()
    )

    return document_question, data_question

In [45]:
def run_agent(question):

    decision = choose_tool(question)

    print("Tool selected:", decision)

    if decision == "DOCUMENT":

        answer = answer_question(question)

    elif decision == "DATA":

        answer = data_tool(question)

    elif decision == "BOTH":

        document_question, data_question = split_question(question)

        document_answer = answer_question(document_question)

        data_answer = data_tool(data_question)

        prompt = f"""
Answer the original question using both results.

Original question:
{question}

Document result:
{document_answer}

Database result:
{data_answer}

Give one clear and short answer.
Answer in the same language as the original question.
"""

        response = client.chat.completions.create(
            model="openrouter/free",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        answer = response.choices[0].message.content

    else:

        answer = "I cannot answer this question."

    return answer

In [46]:
# DOCUMENT
run_agent("Quanto costa la sosta nella ZTL?")

Tool selected: DOCUMENT


"La sosta nella ZTL Centrale costa **€ 2,80 l'ora**."

In [47]:
# DATA
run_agent("Qual è la velocità media del traffico?")

Tool selected: DATA
Generated SQL:
SELECT AVG(speed) FROM traffic_locations;


[{'avg': 55.83133928571429}]

In [48]:
# BOTH
run_agent("""
Quanto costa la sosta nella ZTL
e qual è la velocità media del traffico?
""")

Tool selected: User Safety: safe


'I cannot answer this question.'

In [49]:
# DOCUMENT - should use RAG
run_agent("Quali sono gli orari della sosta a pagamento?")

Tool selected: DOCUMENT


'\n\nLa sosta a pagamento è in vigore dal lunedì al sabato, dalle ore 8:00 alle ore 19:30. In alcune aree ospedaliere (Oftalmico e Valdese), il termine è anticipato alle ore 18:30.'